# 02 — LSTM Model Development

> **Portfolio Optimizer** · *Deep Learning Price Forecasting*

**Objectives**
1. Engineer 18 technical features used as LSTM inputs.
2. Build and train a 3-layer LSTM with MC-Dropout uncertainty estimation.
3. Tune lookback window, learning rate, and dropout via grid search.
4. Evaluate on out-of-sample test set — report RMSE, MAPE, Directional Accuracy.
5. Visualise learning curves, error distributions, and 95% confidence bands.
6. Log everything to MLflow.

In [ ]:
import warnings, os, sys
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import tensorflow as tf
import mlflow

NAVY, SLATE, ORANGE, GREY = '#1A273A', '#3E4A62', '#C24D2C', '#D9D9D7'
def base_layout(**kw):
    return dict(paper_bgcolor=NAVY, plot_bgcolor=NAVY,
                font=dict(color=GREY), xaxis=dict(gridcolor=SLATE),
                yaxis=dict(gridcolor=SLATE), **kw)

TICKER  = 'TCS.NS'
LOOKBACK = 60
EPOCHS   = 50
BATCH    = 32
TEST_SPLIT = 0.15
print(f'TF version: {tf.__version__}  |  GPU: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ── Load processed features ──────────────────────────────────────────────────
from data.data_ingestion import load_processed

df = load_processed(TICKER)
print(f'Shape: {df.shape}  |  Columns: {df.columns.tolist()}')
df.tail(3)

In [ ]:
# ── Prepare LSTM dataset ─────────────────────────────────────────────────────
from models.lstm_model import LSTMDataset
from sklearn.model_selection import train_test_split

FEATURE_COLS = [
    'Close','Open','High','Low','Volume','Returns','SMA_20','SMA_50',
    'EMA_12','RSI_14','MACD','MACD_Signal','BB_Upper','BB_Lower',
    'Vol_10','Vol_20','OBV','Lag_1',
]

dataset = LSTMDataset(df, feature_cols=FEATURE_COLS, lookback=LOOKBACK, target_col='Close')
split   = int(len(dataset) * (1 - TEST_SPLIT))
train_ds, test_ds = dataset[:split], dataset[split:]
print(f'Train sequences: {len(train_ds[0])}  |  Test sequences: {len(test_ds[0])}')

In [ ]:
# ── Build LSTM model ─────────────────────────────────────────────────────────
from models.lstm_model import build_lstm_model

model = build_lstm_model(
    input_shape=(LOOKBACK, len(FEATURE_COLS)),
    lstm_units=[128, 64, 32],
    dropout_rate=0.2,
    dense_units=16,
)
model.summary()

In [ ]:
# ── Train with MLflow logging ────────────────────────────────────────────────
from models.lstm_model import train_lstm
from mlops.mlflow_tracking import initialise_mlflow

mlflow.set_tracking_uri('sqlite:///../mlflow_server/mlflow.db')
initialise_mlflow(experiment_name='LSTM_Development')

trained_model, history, scaler, metrics = train_lstm(
    df=df,
    ticker=TICKER,
    feature_cols=FEATURE_COLS,
    lookback=LOOKBACK,
    epochs=EPOCHS,
    batch_size=BATCH,
    test_split=TEST_SPLIT,
    log_to_mlflow=True,
    experiment_name='LSTM_Development',
)

print('\n── Test Metrics ─────────────────────────────────────────────')
for k, v in metrics.items():
    print(f'  {k:30s}: {v:.4f}')

In [ ]:
# ── Training loss curves ─────────────────────────────────────────────────────
hist_df = pd.DataFrame(history.history)

fig = go.Figure()
fig.add_trace(go.Scatter(y=hist_df['loss'],     name='Train Loss', line=dict(color=ORANGE)))
fig.add_trace(go.Scatter(y=hist_df['val_loss'], name='Val Loss',   line=dict(color=GREY, dash='dash')))
fig.update_layout(**base_layout(title=f'{TICKER} — LSTM Training & Validation Loss', height=400,
                                  xaxis_title='Epoch', yaxis_title='Huber Loss'))
fig.show()

In [ ]:
# ── Forecast with 95% confidence interval (MC-Dropout) ───────────────────────
from models.lstm_model import predict_next_n_days

forecast = predict_next_n_days(
    model=trained_model,
    df=df,
    feature_cols=FEATURE_COLS,
    scaler=scaler,
    lookback=LOOKBACK,
    n_days=30,
    mc_passes=100,
)

forecast_df = pd.DataFrame(forecast)
print(f'Forecast shape: {forecast_df.shape}')
forecast_df.head(5)

In [ ]:
# ── Visualise 30-day forecast with CI ────────────────────────────────────────
hist_close = df['Close'].tail(60)

fig = go.Figure()
fig.add_trace(go.Scatter(x=hist_close.index, y=hist_close, name='Historical',
                          line=dict(color=GREY, width=2)))
fig.add_trace(go.Scatter(x=forecast_df['date'], y=forecast_df['forecast'],
                          name='LSTM Forecast', line=dict(color=ORANGE, width=2)))
fig.add_trace(go.Scatter(
    x=pd.concat([forecast_df['date'], forecast_df['date'][::-1]]),
    y=pd.concat([forecast_df['upper_95'], forecast_df['lower_95'][::-1]]),
    fill='toself', fillcolor=f'rgba(194,77,44,0.15)',
    line=dict(color='rgba(0,0,0,0)'), name='95% CI',
))

fig.update_layout(**base_layout(title=f'{TICKER} — 30-Day LSTM Forecast + 95% CI', height=500))
fig.show()

In [ ]:
# ── Hyperparameter sensitivity: lookback window ──────────────────────────────
lookbacks = [20, 40, 60, 80, 100]
mape_scores = {}

for lb in lookbacks:
    try:
        _, _, _, m = train_lstm(df, TICKER, FEATURE_COLS, lookback=lb, epochs=15,
                                 batch_size=32, test_split=0.15, log_to_mlflow=False)
        mape_scores[lb] = m.get('mape', np.nan)
        print(f'Lookback={lb:3d}  MAPE={mape_scores[lb]:.2f}%')
    except Exception as e:
        print(f'Lookback={lb:3d}  Error: {e}')

fig = go.Figure(go.Scatter(
    x=list(mape_scores.keys()), y=list(mape_scores.values()),
    mode='lines+markers', line=dict(color=ORANGE, width=2),
    marker=dict(size=8, color=ORANGE),
))
fig.update_layout(**base_layout(title='MAPE vs Lookback Window', height=400,
                                  xaxis_title='Lookback (days)', yaxis_title='MAPE (%)'))
fig.show()

In [ ]:
print('\n✅  LSTM Development complete.')
print(f'   Best model MAPE  : {metrics["mape"]:.2f}%')
print(f'   Directional Acc  : {metrics["directional_accuracy"]*100:.1f}%')
print(f'   Saved checkpoint : checkpoints/{TICKER.replace(".","_")}_best.h5')